# Bollinger Band Breakout — Backtest

Runs BBB strategy on XAUUSD 5-minute bars using NautilusTrader BacktestEngine.
Output goes to `backtest_catalog/` as feather files for the dashboard.

In [ ]:
import os
from decimal import Decimal
from pathlib import Path

import pandas as pd

from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import InstrumentId, Symbol, TraderId, Venue
from nautilus_trader.model.objects import Currency, Money, Price, Quantity
from nautilus_trader.persistence.config import StreamingConfig
from nautilus_trader.persistence.wranglers import BarDataWrangler

from runner.strategies.bbb_strategy import (
    ArrayKind,
    BandKind,
    BBBSignalVariant,
    BBBStrategy,
    BBBStrategyConfig,
    MATrendKind,
)

In [ ]:
SIM = Venue("SIM")

from nautilus_trader.model.instruments import CurrencyPair

XAUUSD_SIM = CurrencyPair(
    instrument_id=InstrumentId(Symbol("XAU/USD"), SIM),
    raw_symbol=Symbol("XAU/USD"),
    base_currency=Currency.from_str("XAU"),
    quote_currency=Currency.from_str("USD"),
    price_precision=2,
    size_precision=0,
    price_increment=Price.from_str("0.01"),
    size_increment=Quantity.from_int(1),
    lot_size=None,
    max_quantity=None,
    min_quantity=Quantity.from_int(1),
    max_price=None,
    min_price=None,
    margin_init=Decimal("0.03"),
    margin_maint=Decimal("0.03"),
    maker_fee=Decimal("0.00002"),
    taker_fee=Decimal("0.00002"),
    ts_event=0,
    ts_init=0,
)

bar_type = BarType.from_str("XAU/USD.SIM-5-MINUTE-BID-EXTERNAL")
print(f"Instrument: {XAUUSD_SIM.id}")
print(f"Bar type: {bar_type}")

In [ ]:
# Load XAUUSD 5-minute bar data from CSV
# Expected columns: timestamp (or datetime), open, high, low, close, volume
# Adjust path to your data source
DATA_PATH = Path("../../../data/xauusd_5m.csv")  # Adjust this path

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.set_index("timestamp")
df.index = pd.to_datetime(df.index, utc=True)

print(f"Loaded {len(df)} bars from {df.index[0]} to {df.index[-1]}")
df.head()

In [ ]:
wrangler = BarDataWrangler(bar_type=bar_type, instrument=XAUUSD_SIM)
bars = wrangler.process(df)
print(f"Wrangled {len(bars)} bars")

In [ ]:
CATALOG_PATH = os.environ.get(
    "NAUTILUS_STORE_PATH",
    str(Path.home() / "code" / "nautilus_trader" / "backtest_catalog"),
)

engine_config = BacktestEngineConfig(
    trader_id=TraderId("BACKTESTER-001"),
    logging=LoggingConfig(log_level="INFO"),
    streaming=StreamingConfig(
        catalog_path=CATALOG_PATH,
        replace_existing=True,
    ),
)

engine = BacktestEngine(config=engine_config)

engine.add_venue(
    venue=SIM,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    base_currency=USD,
    starting_balances=[Money(100_000, USD)],
)

engine.add_instrument(XAUUSD_SIM)
engine.add_data(bars)

# Default config: Buy Top 1SD, Sell Top 3SD (matches Notion analysis)
strategy_config = BBBStrategyConfig(
    instrument_id=XAUUSD_SIM.id,
    bar_type=bar_type,
    trade_size=Decimal("1"),
    buy_array_kind=ArrayKind.CLOSE,
    buy_band_kind=BandKind.TOP,
    buy_period=20,
    buy_sd=1.0,
    sell_array_kind=ArrayKind.CLOSE,
    sell_band_kind=BandKind.TOP,
    sell_period=20,
    sell_sd=3.0,
    frequency_bars=10,
    signal_variant=BBBSignalVariant.BASELINE,
)

strategy = BBBStrategy(config=strategy_config)
engine.add_strategy(strategy=strategy)

print("Running backtest...")
engine.run()
print("Backtest complete!")

In [ ]:
with pd.option_context("display.max_rows", 100, "display.max_columns", None, "display.width", 300):
    print("=== Account Report ===")
    print(engine.trader.generate_account_report(SIM))
    print("\n=== Positions Report ===")
    print(engine.trader.generate_positions_report())
    print("\n=== Order Fills Report ===")
    print(engine.trader.generate_order_fills_report())

In [ ]:
engine.reset()
engine.dispose()
print(f"Results written to {CATALOG_PATH}")